# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`This notebook demonstrates loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library. All references to dataset table structures, fields, and columns are made via their Croissant schema `@id` values to ensure semantic accuracy and reproducibility.### Dataset SourceThe original dataset is defined by a Croissant schema at:`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Install mlcroissant if not present!pip install --quiet mlcroissant

## 1. Data LoadingLoad metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport jsonimport matplotlib.pyplot as plt# Define Croissant schema URLcroissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"# Load dataset and metadatadataset = mlc.Dataset(croissant_url)# Metadata must be accessed as a single object.metadata = dataset.metadataprint(f"{metadata.name}: {metadata.description}")print("Published:", metadata.datePublished)print("Authors (@id):", [author['@id'] for author in metadata.author])

## 2. Data OverviewReview available record sets, their field definitions and IDs.**Note**: All entities are referenced by their `@id`.

In [ ]:
# List all available record set @id's in the datasetrecord_sets = []for obj in dataset.metadata.recordSet:    record_sets.append(obj['@id'])print("Available record sets:")for rs_id in record_sets:    print("- RecordSet @id:", rs_id)# For each record set, print its fields and columns (@id)for rs_obj in dataset.metadata.recordSet:    print(f"\nRecordSet @id: {rs_obj['@id']}")    # List fields    if 'field' in rs_obj:        print("  Fields:")        for fld in rs_obj['field']:            print(f"    - Field @id: {fld['@id']} (name: {fld.get('name', '')})")            if 'column' in fld:                for col in fld['column']:                    print(f"      - Column @id: {col['@id']} (name: {col.get('name', '')})")

## 3. Data ExtractionLoad data from each record set into a Pandas DataFrame for detailed analysis.
Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all available record sets, loading by @iddataframes = {}print("\nLoading each record set as DataFrame by @id:")for record_set_id in record_sets:    records = list(dataset.records(record_set=record_set_id))    df = pd.DataFrame(records)    dataframes[record_set_id] = df    print(f"RecordSet @id: {record_set_id}")    print("Columns:", df.columns.tolist())    print(df.head(), "\n")# Choose the first available record set for demonstrationselected_record_set_id = record_sets[0] if len(record_sets) > 0 else Noneif selected_record_set_id:    print("Sample data from record set @id:", selected_record_set_id)    print(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps: filtering records, normalizing numeric fields, grouping by a categorical field.
All fields referenced by `@id`.

In [ ]:
# Select a numeric field from the selected record set by @idif selected_record_set_id:    df = dataframes[selected_record_set_id]    numeric_fields = [col for col in df.columns if df[col].dtype != 'O' and 'age' in col.lower()]    if numeric_fields:        numeric_field_id = numeric_fields[0]  # e.g. '@id' for 'age_at_diagnosis'        print(f"Numeric field selected for EDA: {numeric_field_id}")        threshold = 18  # Example threshold for age        filtered_df = df[df[numeric_field_id] > threshold]        print(f"Filtered records with {numeric_field_id} > {threshold}:")        print(filtered_df.head())        norm_col = f"{numeric_field_id}_normalized"        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()        print(f"Normalized {numeric_field_id} for filtered records:")        print(filtered_df[[numeric_field_id, norm_col]].head())        # Select a grouping field - e.g. 'infertility' as a categorical field if available        group_field_id = [col for col in df.columns if 'infertility' in col.lower() or 'phenotype' in col.lower()]        if group_field_id:            group_field = group_field_id[0]            print(f"Grouping by field @id: {group_field}")            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)            print(f"Grouped data by {group_field}:")            print(grouped_df.head())    else:        print("No numeric field found in record set for demonstration EDA.")

## 5. VisualizationVisualize distributions or relationships between fields in the dataset. All fields referenced by `@id`.

In [ ]:
# Basic histogram and scatter plot for numeric fieldsif selected_record_set_id and numeric_fields:    df = dataframes[selected_record_set_id]    plt.figure(figsize=(8, 4))    plt.hist(df[numeric_field_id].dropna(), bins=5, color='skyblue', edgecolor='black')    plt.title(f"Distribution of {numeric_field_id}")    plt.xlabel(numeric_field_id)    plt.ylabel("Count")    plt.show()    # If there's a second numeric field (e.g. hormone level), plot correlation    other_numeric_fields = [col for col in df.columns if df[col].dtype != 'O' and col != numeric_field_id]    if other_numeric_fields:        other_field_id = other_numeric_fields[0]        plt.figure(figsize=(6, 6))        plt.scatter(df[numeric_field_id], df[other_field_id], c='green')        plt.title(f"Scatter: {numeric_field_id} vs {other_field_id}")        plt.xlabel(numeric_field_id)        plt.ylabel(other_field_id)        plt.show()

## 6. ConclusionThis notebook demonstrated loading, overview, and exploratory analysis of the FAIR^2 clinical dataset for NC-21OHD-associated infertility using only Croissant schema `@id`s for all references.- The dataset is small and heterogeneous, with only three cases, and thus optimal for benchmarking and clinical illustration rather than ML model development.
- Each table, field, and column can be accessed via its `@id`, supporting reproducible multi-table research.
- The data is ready for further clinical data enrichment or comparative analysis upon extension.